# Reference Condition Distribution Analysis

Analyze dataset density across the three dimensions of **Iref / Tref / OHref**. Objective: find reference conditions with the highest concentration of data points, maximizing available ground-truth values near each reference.

**Analysis Process**

1. Load the Parquet dataset and use `GMpreprocess` for column mapping and unit conversion.

2. Use `Urc1.preprocess_once` to perform model-consistent filtering (current/voltage/temperature/OH windows).

3. Plot three one-dimensional distributions (I, T, OH).

4. Plot a three-dimensional histogram (I x T x OH), where high-density bins indicate candidate reference positions.

5. Output a top-k hotspot table with recommended Iref / Tref / OHref values.


In [ ]:
import sys, os

# Ensure the project root directory is in sys.path (regardless of launch directory).
_PROJECT_ROOT = os.path.abspath(os.path.join(os.path.dirname("__file__"), "..", "..", ".."))
if _PROJECT_ROOT not in sys.path:
    sys.path.insert(0, _PROJECT_ROOT)

from master_arbeit_Di.ground_truth.a_1_data_distribution import (
    analyze_reference_condition_distribution,
    PreprocessConfig,
    HistogramConfig,
)

print("Import OK - project root:", _PROJECT_ROOT)


---
## Common Configuration

Preprocessing parameters and plotting binning parameters are only entered here; all datasets share the same configuration. If a dataset requires different filtering ranges, these can be overridden separately at runtime.


In [ ]:
# ================================================================
# Common preprocessing parameters (keep consistent with Urc1 training settings)
# ================================================================
PREPROCESS_CFG = PreprocessConfig(
    i_off=0.1,
    u_off=1.3,
    resample=1,
    data_filter_i_min=0.1,
    data_filter_U_min=1.4,
    data_filter_U_max=2.3,
    data_filter_T_min=50.0,
    data_filter_T_max=65.0,
    data_filter_h_since_last_start_min=0.5,
)

# ================================================================
# Common binning parameters
# bins_i / bins_t / bins_oh: number of bins for 1D histograms
# bins_3d_i / bins_3d_t / bins_3d_oh: number of bins for 3D histogram
# use_log_oh_for_3d: recommended when OH spans a wide range
# min_count_for_3d: only show bins with count >= this threshold
# top_k_hotspots: show top-k bins in hotspot table
# ================================================================
HIST_CFG = HistogramConfig(
    bins_i=80,
    bins_t=60,
    bins_oh=80,
    bins_3d_i=25,
    bins_3d_t=20,
    bins_3d_oh=20,
    use_log_oh_for_3d=True,
    min_count_for_3d=10,
    top_k_hotspots=20,
)

# ================================================================
# Output directories for HTML and CSV results
# ================================================================
PREPROCESS_OUT = r"explore_data\output"
ANALYSIS_OUT   = r"output_backup\output_histrogram"

print("Config ready.")


---
## Dataset 1 - G6M2


In [ ]:
G6M2_RESULT = analyze_reference_condition_distribution(
    dataset_path=r"..\\..\\explore_data\\G6M2.parquet",
    preprocess_output_dir=PREPROCESS_OUT,
    analysis_output_dir=ANALYSIS_OUT,
    preprocess_config=PREPROCESS_CFG,
    hist_config=HIST_CFG,
    save_html=True,
    save_csv=True,
)


### G6M2 - One-dimensional distribution (Iref / Tref / OHref)


In [ ]:
G6M2_RESULT["fig_1d"].show()


### G6M2 - 3D Histogram (Iref x Tref x OHref)

Bubble size = number of data points inside each bin. Rotate interactively to inspect high-density regions in 3D space.


In [ ]:
G6M2_RESULT["fig_3d"].show()


### G6M2 - High-Density Reference Condition Candidates (Top Hotspots)

`count` = number of preprocessed points inside this bin that pass model filtering. Larger `OHref` can indicate later operation phases with denser data.

In [ ]:

G6M2_RESULT["hotspot_df"].style \
    .format({"Iref": "{:.4f}", "Tref": "{:.2f}", "OHref": "{:.2f}"}) \
    .background_gradient(subset=["count"], cmap="YlOrRd") \
    .set_caption("G6M2 - Top Reference Condition Hotspots")


### G6M2 - Quantile Statistics

Quickly inspect the post-filter distribution range of I/T/OH to assess whether candidate reference values are reasonable.


In [ ]:
G6M2_RESULT["quantile_summary"].style \
    .format("{:.4f}") \
    .background_gradient(cmap="Blues") \
    .set_caption("G6M2 - Quantile Summary (10th / 25th / 50th / 75th / 90th percentile)")


---
## Dataset 2 - G1M1


In [ ]:
G1M1_RESULT = analyze_reference_condition_distribution(
    dataset_path=r"..\\..\\explore_data\\G1M1_new.parquet",
    preprocess_output_dir=PREPROCESS_OUT,
    analysis_output_dir=ANALYSIS_OUT,
    preprocess_config=PREPROCESS_CFG,
    hist_config=HIST_CFG,
    save_html=True,
    save_csv=True,
)


### G1M1 - One-dimensional distribution (Iref / Tref / OHref)


In [ ]:
G1M1_RESULT["fig_1d"].show()


### G1M1 - 3D Histogram (Iref x Tref x OHref)


In [ ]:
G1M1_RESULT["fig_3d"].show()


### G1M1 - High-Density Reference Condition Candidate


In [ ]:
G1M1_RESULT["hotspot_df"].style \
    .format({"Iref": "{:.4f}", "Tref": "{:.2f}", "OHref": "{:.2f}"}) \
    .background_gradient(subset=["count"], cmap="YlOrRd") \
    .set_caption("G1M1 - Top Reference Condition Hotspots")


In [ ]:
G1M1_RESULT["quantile_summary"].style \
    .format("{:.4f}") \
    .background_gradient(cmap="Blues") \
    .set_caption("G1M1 - Quantile Summary (10th / 25th / 50th / 75th / 90th percentile)")


--- 
## Comparison of Two Datasets

By viewing the density of the two datasets side-by-side under the same set of candidate refs, it becomes easier to select a reference condition that is valid for both datasets.


In [ ]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

def _compare_1d(results: dict, variable: str, axis_label: str, bins: int = 80) -> go.Figure:
    """Overlay 1-D histograms for all datasets for a single variable."""
    colors = ["#1f77b4", "#d62728", "#2ca02c", "#ff7f0e"]
    fig = go.Figure()
    for idx, (ds_name, res) in enumerate(results.items()):
        fig.add_trace(
            go.Histogram(
                x=res["reference_df"][variable],
                nbinsx=bins,
                name=ds_name,
                opacity=0.65,
                marker_color=colors[idx % len(colors)],
            )
        )
    fig.update_layout(
        title=f"{variable} Distribution - Dataset Comparison",
        xaxis_title=axis_label,
        yaxis_title="Count",
        barmode="overlay",
        template="plotly_white",
        height=400,
    )
    return fig


DATASET_RESULTS = {
    "G6M2": G6M2_RESULT,
    "G1M1": G1M1_RESULT,
}

_compare_1d(DATASET_RESULTS, "Iref",  "Iref [A/cm2]").show()
_compare_1d(DATASET_RESULTS, "Tref",  "Tref [degC]").show()
_compare_1d(DATASET_RESULTS, "OHref", "OHref [h]").show()


---
## Add More Datasets (Template)

Copy the cell below, modify `dataset_path` and the variable name, and then run it to analyze any new dataset. All results (HTML/CSV) are automatically saved to the `ANALYSIS_OUT` directory.

In [ ]:
# ====== New dataset template: update path and run ======

# NEW_RESULT = analyze_reference_condition_distribution(
#     dataset_path=r"explore_data\YOUR_DATASET.parquet",
#     preprocess_output_dir=PREPROCESS_OUT,
#     analysis_output_dir=ANALYSIS_OUT,
#     preprocess_config=PREPROCESS_CFG,
#     hist_config=HIST_CFG,
#     save_html=True,
#     save_csv=True,
# )
# NEW_RESULT["fig_1d"].show()
# NEW_RESULT["fig_3d"].show()
# NEW_RESULT["hotspot_df"]
